# M4.13 · Seq2seq, Bahdanau attention ve beam search

Sinirsel makine çevirisinin (NMT) kodlayıcı–kod çözücü mimarisi: tüm kaynak cümleyi tek bir vektöre sıkıştıran RNN, öğretmen zorlaması (teacher forcing), EOS, dolgu ve maskeleme, uzunluğa göre kovalama, büyük çıktı sözlüğü için örneklenmiş softmax, zamanlanmış örnekleme ve çift yönlü RNN'ler. Açgözlü çözümlemenin hatası ve beam search (uzunluk normalizasyonuyla), Bahdanau (toplamsal) ve Luong (çarpımsal) attention. Saf Python'da beam search, NumPy'da maskeli toplamsal/çarpımsal attention ve maskeli kayıp, PyTorch'ta tarih biçimi çevirisinde attention'sız ve attention'lı seq2seq ile hizalama haritası, sonda toplu (batched) beam search. Ekler: BLEU, kovalama, örneklenmiş softmax'ın yanlılığı.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-13-seq2seq.html](https://m-gonul.github.io/pytorch-lab/m4-13-seq2seq.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_13_seq2seq.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\nTREE = {(0,): {\"A\": 0.6, \"B\": 0.4}, (0, \"A\"): {\"x\": 0.4, \"y\": 0.3, \"z\": 0.3}, (0, \"B\"): {\"x\": 0.9, \"y\": 0.1}}\n\n\ndef tree_step(prefix):\n    if prefix in TREE:\n        return TREE[prefix]\n    return {\"<eos>\": 1.0}\n\n\ndef _spread(d, prefix, n):\n    rest = (1 - sum(d.values())) / n\n    d.update({f\"{prefix}{i}\": rest for i in range(n)})\n    return d\n\n\nHOW = {(): _spread({\"How\": 0.75, \"What\": 0.03, \"You\": 0.01}, \"x\", 21), (\"How\",): _spread({\"will\": 0.36, \"are\": 0.32, \"do\": 0.16}, \"h\", 8),\n       (\"What\",): {\"are\": 0.5, \"<eos>\": 0.5}, (\"How\", \"will\"): _spread({\"you\": 0.074}, \"w\", 20), (\"How\", \"are\"): _spread({\"you\": 5 / 12}, \"a\", 30),\n       (\"How\", \"do\"): _spread({\"you\": 2 / 3}, \"d\", 20), (\"How\", \"are\", \"you\"): _spread({\"<eos>\": 0.6, \"doing\": 0.3}, \"r\", 10),\n       (\"How\", \"do\", \"you\"): {\"do\": 0.875, \"<eos>\": 0.125}}\n\n\ndef how_step(prefix):\n    return HOW.get(tuple(prefix[1:]), {\"<eos>\": 1.0})\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t_tree():\n        s, lp = need(ns, \"greedy_decode\")(tree_step, 0, \"<eos>\", 5)\n        ensure(s == [\"A\", \"x\", \"<eos>\"], f\"açgözlü {s}\")\n        close(math.exp(lp), 0.24, what=\"açgözlü olasılık\")\n        res = need(ns, \"beam_search\")(tree_step, 0, \"<eos>\", 2, 5)\n        ensure(res[0][0] == [\"B\", \"x\", \"<eos>\"], f\"beam {res[0][0]}\")\n        close(math.exp(res[0][1]), 0.36, what=\"beam olasılığı\")\n    R_.test(\"açgözlü vs beam\", t_tree)\n\n    def t_how():\n        g = ns[\"greedy_decode\"](how_step, \"<s>\", \"<eos>\", 6)[0]\n        rows = [[\"açgözlü\", \" \".join(g), f\"{math.exp(ns['greedy_decode'](how_step, '<s>', '<eos>', 6)[1]):.3f}\"]]\n        for k, a in ((1, 0.0), (3, 0.0), (3, 1.0)):\n            r = ns[\"beam_search\"](how_step, \"<s>\", \"<eos>\", k, 6, a)\n            rows.append([f\"beam k = {k}, α = {a}\", \" \".join(r[0][0]), f\"{math.exp(r[0][1]):.3f}\"])\n        ensure(g == [\"How\", \"will\", \"you\", \"<eos>\"], f\"açgözlü 'How will you' tuzağına düşer: {g}\")\n        r3 = ns[\"beam_search\"](how_step, \"<s>\", \"<eos>\", 3, 6, 1.0)\n        ensure(r3[0][0][:3] == [\"How\", \"are\", \"you\"] or r3[0][0][:3] == [\"How\", \"do\", \"you\"], f\"beam + normalizasyon: {r3[0][0]}\")\n        ensure(ns[\"beam_search\"](how_step, \"<s>\", \"<eos>\", 1, 6)[0][0] == g, \"k = 1 = açgözlü\")\n        R_.table(\"'Comment vas-tu?' oyuncak modeli\", [\"çözümleyici\", \"çıktı\", \"olasılık\"], rows,\n                 note=\"Normalizasyonsuz beam search ('How <eos>' gibi) kısa hipotezleri kayırır: her token olasılığı 1'den küçük bir çarpandır. α ile log olasılığı uzunluğa bölmek (Wu ve ark. 2016: ((5 + L)/6)^α) bu yanlılığı dengeler.\")\n    R_.test(\"Géron örneği\", t_how)\n\n    def t_pad():\n        p, m = need(ns, \"pad_batch\")([[5, 6, 7], [8]], 0)\n        ensure(p == [[5, 6, 7], [8, 0, 0]] and m == [[True] * 3, [True, False, False]], \"dolgu ve maske\")\n        ensure(need(ns, \"shift_right\")([4, 5, 2], 1) == [1, 4, 5], \"öğretmen zorlaması girdisi\")\n    R_.test(\"dolgu ve öğretmen zorlaması\", t_pad)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"masked_softmax\", \"additive_attention\", \"luong_attention\", \"masked_ce\")])\n    rng = np.random.default_rng(0)\n\n    def t():\n        e = np.e\n        a, c = need(ns, \"luong_attention\")(np.array([[1., 0]]), np.array([[[1., 0], [0, 1], [1, 1]]]), np.ones((1, 3), bool))\n        close(a[0], [e / (2 * e + 1), 1 / (2 * e + 1), e / (2 * e + 1)], what=\"el örneği α\")\n        close(c[0, 0], 2 * e / (2 * e + 1), what=\"bağlam\")\n        B, T, dh, ds, da = 3, 6, 5, 4, 7\n        s, H = rng.normal(size=(B, ds)), rng.normal(size=(B, T, dh))\n        W1, W2, v = rng.normal(size=(dh, da)), rng.normal(size=(ds, da)), rng.normal(size=da)\n        mask = np.arange(T)[None] < np.array([6, 3, 1])[:, None]\n        a, c = need(ns, \"additive_attention\")(s, H, W1, W2, v, mask)\n        for b in range(B):\n            L = mask[b].sum()\n            ee = np.array([v @ np.tanh(W1.T @ H[b, i] + W2.T @ s[b]) for i in range(L)])\n            ref = np.exp(ee - ee.max()) / np.exp(ee - ee.max()).sum()\n            close(a[b, :L], ref, atol=1e-12, what=f\"toplamsal α (örnek {b})\")\n            ensure(np.all(a[b, L:] == 0), \"dolgu konumları 0 ağırlık\")\n            close(c[b], ref @ H[b, :L], atol=1e-12, what=\"bağlam\")\n        Wg = rng.normal(size=(ds, dh))\n        a2, _ = ns[\"luong_attention\"](s, H, mask, Wg)\n        sc = np.einsum(\"bd,btd->bt\", s @ Wg, H)\n        close(a2[0], np.exp(sc[0] - sc[0].max()) / np.exp(sc[0] - sc[0].max()).sum(), atol=1e-12, what=\"Luong general\")\n        logits = rng.normal(size=(2, 4, 6))\n        tg = np.array([[3, 4, 2, 0], [5, 0, 0, 0]])\n        lp = logits - np.log(np.exp(logits).sum(-1, keepdims=True))\n        close(need(ns, \"masked_ce\")(logits, tg, 0), -(lp[0, 0, 3] + lp[0, 1, 4] + lp[0, 2, 2] + lp[1, 0, 5]) / 4, what=\"maskeli CE\")\n    R_.test(\"attention ve maskeler\", t)\n",
 "d": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    X, Y = need(ns, \"dates\")(12000, 0)\n    Xt, Yt = ns[\"dates\"](1000, 1)\n    x, y = need(ns, \"encode\")(X, Y)\n    xt, yt = ns[\"encode\"](Xt, Yt)\n\n    def t():\n        res, rows, models = {}, [], {}\n        for attn in (False, True):\n            torch.manual_seed(0)\n            m = need(ns, \"Seq2Seq\")(64, attn)\n            out = m(x[:4], y[:4])\n            ensure(tuple(out.shape) == (4, 11, len(ns[\"OUT\"]) + 3), \"logit şekli (B, T, V)\")\n            opt = torch.optim.Adam(m.parameters(), 3e-3)\n            t0 = time.perf_counter()\n            for ep in range(2):\n                perm = torch.randperm(len(x))\n                for i in range(0, len(x), 64):\n                    idx = perm[i:i + 64]\n                    loss = F.cross_entropy(m(x[idx], y[idx]).flatten(0, 1), y[idx].flatten())\n                    opt.zero_grad()\n                    loss.backward()\n                    opt.step()\n            m.eval()\n            p = m.greedy(xt)\n            res[attn] = (p == yt).all(1).float().mean().item()\n            tokacc = (p == yt).float().mean().item()\n            models[attn] = m\n            rows.append([\"Bahdanau attention'lı\" if attn else \"attention'sız (yalnız son durum)\", sum(q.numel() for q in m.parameters()), f\"{tokacc:.3f}\", f\"{res[attn]:.3f}\", f\"{time.perf_counter() - t0:.0f} sn\"])\n        ensure(res[True] > 0.85 and res[True] > res[False] + 0.3, f\"attention farkı {res}\")\n        A = models[True].attention_map(xt[:200])\n        ensure(tuple(A.shape) == (200, 11, xt.shape[1]), \"attention haritası (B, T_out, T_in)\")\n        ensure(torch.allclose(A.sum(-1), torch.ones(200, 11), atol=1e-5) and float(A.masked_select(~(xt[:200] != 0)[:, None].expand_as(A)).abs().max()) == 0.0, \"satırlar 1'e toplanır; dolguya ağırlık yok\")\n        L = (xt[:200] != 0).sum(1)\n        yearmask = (torch.arange(xt.shape[1])[None] >= (L - 4)[:, None]) & (torch.arange(xt.shape[1])[None] < L[:, None])\n        mass = (A[:, :4] * yearmask[:, None]).sum(-1).mean().item()\n        ensure(mass > 0.5, f\"yıl rakamları üretilirken attention girdideki yıla bakar ({mass:.2f})\")\n        R_.table(\"Tarih çevirisi (12 000 örnek, 2 epoch, açgözlü çözümleme)\", [\"model\", \"parametre\", \"token doğruluğu\", \"tam eşleşme\", \"süre\"], rows,\n                 note=f\"Attention'sız modelde kod çözücü tüm girdiyi tek bir 64 boyutlu vektörden çıkarmak zorunda; attention'lı model her çıkış karakterinde ilgili girdi karakterlerine bakar. Yıl rakamları (çıkış 1–4) üretilirken attention kütlesinin ortalama {mass:.0%}'i girdinin son dört karakterinde (yıl) — hizalama hiç denetim olmadan öğrenildi.\")\n    R_.test(\"attention'sız vs Bahdanau\", t)\n",
 "e": "import time\nimport torch\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    torch.manual_seed(0)\n    V, d = 40, 32\n    emb = torch.nn.Embedding(V, d)\n    cell = torch.nn.GRUCell(d, d)\n    head = torch.nn.Linear(d, V)\n    with torch.no_grad():\n        head.weight.mul_(3)\n\n    @torch.no_grad()\n    def step(tok, h):\n        h2 = cell(emb(tok), h)\n        return torch.log_softmax(head(h2), -1), h2\n\n    def t():\n        h0 = torch.randn(6, d)\n        a_t, a_s = need(ns, \"beam_loop\")(step, h0, 0, 1, 3, 8)\n        b_t, b_s = need(ns, \"beam_batched\")(step, h0, 0, 1, 3, 8)\n        ensure(tuple(b_t.shape) == (6, 8), \"(B, T) token\")\n        ensure(torch.equal(a_t, b_t), \"döngü = toplu\")\n        close(a_s.numpy(), b_s.numpy(), atol=1e-4, what=\"skorlar\")\n        g_t, g_s = ns[\"beam_batched\"](step, h0, 0, 1, 1, 8)\n        ensure(bool((b_s >= g_s - 1e-5).all()), \"beam skoru ≥ açgözlü skoru\")\n        rows = []\n        for B, k in ((8, 3), (8, 8), (32, 5)):\n            h0 = torch.randn(B, d)\n            r = [B, k]\n            for f in (ns[\"beam_loop\"], ns[\"beam_batched\"]):\n                t0 = time.perf_counter()\n                f(step, h0, 0, 1, k, 15)\n                r.append(fmt_time(time.perf_counter() - t0))\n            rows.append(r)\n        gain = (b_s - g_s).mean().item()\n        R_.table(\"Beam search, T = 15, V = 40 (CPU)\", [\"batch\", \"k\", \"döngü\", \"toplu\"], rows,\n                 note=f\"Toplu sürüm B·k hipotezi tek bir step çağrısıyla ilerletir; aday seçimi (B, k·V) üzerinde tek topk, geri izleme gather ile. Bu örnekte k = 3 beam, açgözlüye göre ortalama {gain:.2f} nat daha yüksek log olasılık buldu.\")\n    R_.test(\"toplu beam search\", t)\n",
 "x1": "import math\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        refs = [\"the cat is on the mat\".split(), \"there is a cat on the mat\".split()]\n        ensure(need(ns, \"modified_precision\")(\"the the the the the the the\".split(), refs, 1) == (2, 7), \"Papineni örneği: 2/7\")\n        ensure(ns[\"modified_precision\"](\"the cat the cat on the mat\".split(), refs, 2) == (4, 6), \"bigram kırpma\")\n        close(need(ns, \"brevity_penalty\")(4, 6), math.exp(1 - 6 / 4), what=\"kısalık cezası\")\n        ensure(ns[\"brevity_penalty\"](7, 6) == 1.0, \"uzun aday cezasız\")\n        close(need(ns, \"bleu\")(\"the cat is on the mat\".split(), refs), 1.0, what=\"birebir eşleşme\")\n        cands = {\"birebir\": \"the cat is on the mat\", \"kelime sırası bozuk\": \"mat the on is cat the\", \"kısa\": \"the cat\", \"eşanlamlı\": \"the cat sits on the mat\", \"the × 7\": \"the the the the the the the\"}\n        rows = []\n        for k, c in cands.items():\n            cc = c.split()\n            ps = [ns[\"modified_precision\"](cc, refs, n) for n in range(1, 5)]\n            rows.append([k, \" / \".join(f\"{a}/{b}\" for a, b in ps), f\"{ns['bleu'](cc, refs, 2):.3f}\", f\"{ns['bleu'](cc, refs):.3f}\"])\n        ensure(ns[\"bleu\"](\"the cat\".split(), refs, 2) < 0.4, \"kısa aday cezalandırılır\")\n        R_.table(\"BLEU (iki referans)\", [\"aday\", \"p₁ … p₄\", \"BLEU-2\", \"BLEU-4\"], rows,\n                 note=\"Kırpma 'the the the…' hilesini, kısalık cezası ('the cat' — yüksek kesinlik, düşük kapsama) kısa çıktıları engeller. BLEU eşanlamlıyı ödüllendirmez ve cümle düzeyinde gürültülüdür (tek bir 4-gram eksik → 0): derlem düzeyinde sayımlar toplanarak hesaplanır.\")\n    R_.test(\"BLEU\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"padding_waste\")([3, 1, 2, 2], [np.array([0, 1]), np.array([2, 3])]), 1 - 8 / 10, what=\"israf tanımı\")\n        rng = np.random.default_rng(0)\n        L = np.clip(rng.lognormal(2.7, 0.6, 20000).astype(int), 2, 200)\n        rb = need(ns, \"random_batches\")(len(L), 64, rng)\n        bb = need(ns, \"bucket_batches\")(L, 64, rng, 50)\n        ensure(sorted(np.concatenate(bb).tolist()) == list(range(len(L))), \"her örnek tam bir kez\")\n        wr, wb = ns[\"padding_waste\"](L, rb), ns[\"padding_waste\"](L, bb)\n        ensure(wb < wr / 3, f\"kovalama israfı azaltır ({wr:.2f} → {wb:.2f})\")\n        rows = [[\"rastgele batch'ler\", f\"{wr:.1%}\"]]\n        for ch in (1, 5, 50, 1000):\n            rows.append([f\"kovalama, parça = {ch} batch\", f\"{ns['padding_waste'](L, ns['bucket_batches'](L, 64, rng, ch)):.1%}\"])\n        R_.table(\"20 000 cümle (log-normal uzunluk, ort. ≈ 18 token), batch 64\", [\"yöntem\", \"dolgu token oranı\"], rows,\n                 note=\"Dolgu token'ları hem bellek hem hesap harcar (RNN'de her adım, Transformer'da karesel). Kovalama benzer uzunlukları birlikte koyar; parça büyüdükçe israf azalır ama batch'lerin içeriği daha az rastgele olur (aynı batch'te hep benzer uzunluklar) — parça boyutu bu ödünleşimi ayarlar.\")\n    R_.test(\"kovalama\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        V, d = 2000, 16\n        W = rng.normal(0, 0.6, (V, d))\n        h = rng.normal(size=d)\n        t_ = 7\n        z = W @ h\n        p = np.exp(z - z.max())\n        p /= p.sum()\n        g = need(ns, \"full_grad\")(h, W, t_)\n        close(g, W.T @ p - W[t_], atol=1e-10, what=\"tam gradyan\")\n        eps = 1e-6\n        e0 = np.zeros(d)\n        e0[3] = eps\n        loss = lambda hh: -(W @ hh)[t_] + np.log(np.exp(W @ hh).sum())\n        close((loss(h + e0) - loss(h)) / eps, g[3], atol=1e-4, what=\"sayısal\")\n        q = (1.0 / np.arange(1, V + 1))\n        q /= q.sum()\n        logq = np.log(q)\n        rows = []\n        errs = {}\n        for m in (20, 100, 500):\n            for corr in (False, True):\n                acc = np.zeros(d)\n                R = 300\n                for _ in range(R):\n                    S = rng.choice(V, m, p=q)\n                    S = S[S != t_]\n                    acc += need(ns, \"sampled_grad\")(h, W, t_, S, logq, corr)\n                err = np.linalg.norm(acc / R - g) / np.linalg.norm(g)\n                errs[(m, corr)] = err\n                rows.append([m, \"evet\" if corr else \"hayır\", f\"{err:.3f}\"])\n        ensure(errs[(500, True)] < errs[(500, False)] and errs[(500, True)] < 0.25, \"log Q düzeltmesi yanlılığı azaltır\")\n        R_.table(\"V = 2000, Zipf önerisi: 300 örneklenmiş gradyanın ortalamasının tam gradyana göreli hatası\", [\"örnek sayısı\", \"log Q düzeltmesi\", \"göreli hata\"], rows,\n                 note=\"Örneklenmiş softmax, tam normalizasyonu yalnız hedef + m örnek üzerinden yapar. Öneri dağılımı düzgün değilse (sık kelimeler daha çok örneklenir) logit'lerden log q çıkarmak önem örneklemesi düzeltmesidir (Jean ve ark. 2015); düzeltmesiz tahminci sık kelimelere doğru yanlıdır. m büyüdükçe her iki hata da azalır.\")\n    R_.test(\"örneklenmiş softmax\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_13_seq2seq as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**NMT için kodlayıcı–kod çözücü** (Sutskever ve ark. 2014): bir RNN kaynak cümleyi okur, diğeri hedef cümleyi üretir. Eğitimde kod çözücüye her adımda bir önceki **doğru** kelime verilir (**öğretmen zorlaması**); çıkarımda kendi son çıktısı. Kod çözücü cümleyi EOS ile bitirir; kaynak cümleler ters çevrilerek verilirse ilk kelimeler kod çözücüye en yakın olur. Kelimeler önce embedding'e çevrilir; çıkışta her adımda sözlük üzerinde softmax.

**Pratik ayrıntılar:** uzunluklar farklıdır → dolgu (padding) + **maskeleme**: (1) kayıpta dolgu ve EOS sonrası token'lar sayılmaz, (2) attention'da dolgu konumları ağırlık almaz. Benzer uzunluktaki cümleleri aynı kovaya koyup batch'leri kova içinden seçmek dolgu israfını azaltır. 50 000 kelimelik çıkış sözlüğünde her adımda softmax pahalıdır: hiyerarşik softmax ya da **örneklenmiş softmax/negatif örnekleme** (yalnız doğru kelime + ~100 rastgele kelime üzerinden normalizasyon; çıkarımda kullanılamaz). **Zamanlanmış örnekleme** (Bengio ve ark. 2015): eğitimde gerçek önceki token'dan modelin kendi çıktısına kademeli geçiş. **Çift yönlü RNN**: "the Queen of the United Kingdom / of hearts / bee" — "queen"i kodlamak için ileriye bakmak gerekir; soldan sağa ve sağdan sola iki katman, çıkışlar birleştirilir.

**Beam search:** açgözlü çözümleme "Comment vas-tu?" için "How will you?" üretebilir — "How will" sonrasında geri dönüş yoktur. Beam search k en olası kısmi cümleyi tutar; her adımda her birini sözlük boyunca uzatır (k × V aday), olasılıkları çarpar ve en iyi k'yı tutar. Ek eğitim olmadan performans artar.

**Attention** (Bahdanau ve ark. 2015): kod çözücü her adımda tüm kodlayıcı çıktılarının ağırlıklı ortalamasını (bağlam) kullanır; $\alpha_{(t,i)}$ ağırlıkları, kodlayıcı çıktısı ile kod çözücünün önceki gizli durumunun birleştirilip tek nöronlu yoğun katmandan geçirilmesiyle (enerji $e_{(t,i)}$) ve softmax'la bulunur — **toplamsal / birleştirmeli** attention. n kelimelik girdi ve çıktı için ~n² ağırlık. **Luong** (2015): skor = nokta çarpım (ya da önce doğrusal dönüşüm: "general"), kod çözücünün **şimdiki** durumu kullanılır, bağlam gizli durumla birleştirilip çıktıya gider — **çarpımsal** attention.

## 1 · Elle

Beam: 0.75 · 0.32 = 0.24 ("How will" 0.27, "How do" 0.12). Her adımda 3 × 10 000 = 30 000 aday puanlanır, en iyi 3 tutulur.

Ağaç: açgözlü önce A'yı (0.6) seçer, sonra x'i (0.4): 0.24. k = 2: ilk adımda {A 0.6, B 0.4}; ikinci adımda adaylar Ax 0.24, Ay 0.18, Az 0.18, Bx 0.36 → en iyi **Bx = 0.36**. İlk adımda düşük olasılıklı dal, sonradan kazanır.

Uzunluk: $\log 0.2/4=-0.402$; 2 token'lık 0.3'lük hipotez: $\log0.3/2=-0.602$. Normalizasyonsuz ($\log p$) kısa hipotez kazanır (−1.20 > −1.61): olasılıkların çarpımı her token'da küçüldüğü için beam search kısa çıktıları kayırır.

Luong: skorlar $s\cdot y_i=1,0,1$ → $\alpha=(e,1,e)/(2e+1)=(0.422, 0.155, 0.422)$; bağlam $=\alpha_1[1,0]+\alpha_2[0,1]+\alpha_3[1,1]=[2e,e+1]/(2e+1)$ → ilk bileşen 0.845.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Beam search**

- `howare`: P(How) = 0.75, P(are | How) = 0.32: P("How are")
- `cand`: Beam genişliği 3, sözlük 10 000: bir adımda değerlendirilen aday sayısı
- `greedy`: Ağaç: P(A) = 0.6, P(B) = 0.4; A'dan sonra x 0.4, y 0.3, z 0.3; B'den sonra x 0.9. Açgözlü iki kelimelik dizinin olasılığı
- `beam`: Aynı ağaç, k = 2 beam search sonucu
- `norm`: Uzunluk normalizasyonu (α = 1): 4 token'lık, olasılığı 0.2 olan hipotezin skoru $\log p/L$

**Attention**

- `alpha1`: Luong (nokta çarpım): $s=[1,0]$, kodlayıcı çıktıları $[1,0]$, $[0,1]$, $[1,1]$ → $\alpha_1$
- `ctx`: Aynı örnek: bağlam vektörünün ilk bileşeni
- `n2`: 50 kelimelik girdi, 50 kelimelik çıktı: hesaplanan attention ağırlığı sayısı
- `bi`: Çift yönlü GRU, her yön 128 birim: birleştirilmiş durum boyutu

In [ ]:
cevaplar = dict(
    howare=None,
    cand=None,
    greedy=None,
    beam=None,
    norm=None,
    alpha1=None,
    ctx=None,
    n2=None,
    bi=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"howare":[0.24,1e-9],"cand":[30000,0],"greedy":[0.24,1e-9],"beam":[0.36,1e-9],"norm":[-0.40235947810852507,0.000001],"alpha1":[0.422318798251511,0.000001],"ctx":[0.844637596503022,0.000001],"n2":[2500,0],"bi":[256,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. `step(prefix)` — bir demet (token dizisi) alıp `{token: olasılık}` sözlüğü döndüren model.

- `greedy_decode(step, sos, eos, max_len)` → `(dizi, log olasılık)`; dizi `sos` içermez, `eos` dahil.
- `beam_search(step, sos, eos, k, max_len, alpha=0.0)` → `[(dizi, logp)]` skora ($\log p/L^\alpha$, L = token sayısı) göre azalan: her adımda açık hipotezleri uzat, en iyi k adayı tut; EOS ile bitenleri tamamlanmışlara taşı; k tamamlanmış ya da `max_len` olunca dur; sonuç = tamamlanmışlar (yoksa açık olanlar).
- `pad_batch(seqs, pad=0)` → `(dolgulu, maske)`.
- `shift_right(y, sos)` — öğretmen zorlaması girdisi: `[sos] + y[:-1]`.

In [ ]:
%%writefile stage_b.py
import math


def greedy_decode(step, sos, eos, max_len):
    ...


def beam_search(step, sos, eos, k, max_len, alpha=0.0):
    ...


def pad_batch(seqs, pad=0):
    ...


def shift_right(y, sos):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz.

- `masked_softmax(scores, mask)` — `mask` False olan konumlar 0 ağırlık.
- `additive_attention(s, H, W1, W2, v, mask)` — `s (B, d_s)`, `H (B, T, d_h)`, `W1 (d_h, d_a)`, `W2 (d_s, d_a)`, `v (d_a,)`: $e_{t,i}=v^\top\tanh(W_1^\top h_i+W_2^\top s)$ → `(α (B, T), bağlam (B, d_h))`.
- `luong_attention(s, H, mask, W=None)` — `W=None`: nokta; değilse "general" $s^\top W h_i$.
- `masked_ce(logits, targets, pad=0)` — `(B, T, V)`; dolgu hedefleri hariç ortalama çapraz entropi.

In [ ]:
%%writefile stage_c.py
import numpy as np


def masked_softmax(scores, mask):
    ...


def additive_attention(s, H, W1, W2, v, mask):
    ...


def luong_attention(s, H, mask, W=None):
    ...


def masked_ce(logits, targets, pad=0):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Görev: tarih biçimi çevirisi (Géron'un alıştırması) — `"April 22, 2019"`, `"22 April 2019"`, `"Apr 22 2019"` → `"2019-04-22"`. `dates(n, seed)` ve `encode(X, Y)` verilir (karakter düzeyinde; çıkış sözlüğü: PAD 0, SOS 1, EOS 2, rakamlar ve "-").

- `Seq2Seq(h=64, attn=True)` — embedding (32) + çift yönlü GRU kodlayıcı; başlangıç durumu $\tanh(W[\vec h_T;\overleftarrow h_1])$; `GRUCell` kod çözücü. `attn=True`: Bahdanau — $e=v^\top\tanh(W_1h_i+W_2s_{t-1})$, dolgu maskesi, bağlam GRU girdisine ve çıktı katmanına eklenir.
  - `forward(x, y)` — öğretmen zorlamasıyla logit'ler `(B, T, V)`.
  - `greedy(x, T=11)` — token'lar `(B, T)`.
  - `attention_map(x, T=11)` — açgözlü çözümleme sırasındaki ağırlıklar `(B, T, T_in)` (attn=True).

In [ ]:
%%writefile stage_d.py
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
IN = sorted(set("".join(MONTHS) + "0123456789, "))
OUT = list("0123456789-")
PAD, SOS, EOS = 0, 1, 2


def dates(n, seed):
    r = random.Random(seed)
    X, Y = [], []
    for _ in range(n):
        y, m, d = r.randint(1000, 2999), r.randint(1, 12), r.randint(1, 28)
        X.append([f"{MONTHS[m - 1]} {d}, {y}", f"{d} {MONTHS[m - 1]} {y}", f"{MONTHS[m - 1][:3]} {d:02d} {y}"][r.randint(0, 2)])
        Y.append(f"{y}-{m:02d}-{d:02d}")
    return X, Y


def encode(X, Y):
    iv = {c: i + 1 for i, c in enumerate(IN)}
    ov = {c: i + 3 for i, c in enumerate(OUT)}
    L = max(map(len, X))
    xs = torch.tensor([[iv[c] for c in x] + [PAD] * (L - len(x)) for x in X])
    ys = torch.tensor([[ov[c] for c in y] + [EOS] for y in Y])
    return xs, ys


class Seq2Seq(nn.Module):
    def __init__(self, h=64, attn=True):
        super().__init__()
        ...

    def forward(self, x, y):
        ...

    @torch.no_grad()
    def greedy(self, x, T=11):
        ...

    @torch.no_grad()
    def attention_map(self, x, T=11):
        ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`step(tokens (N,), h (N, d)) → (log olasılıklar (N, V), yeni h)` — rastgele başlatılmış küçük bir GRU dil modeli. Kurallar (eşitlik testi için): tam `T` adım; EOS üretmiş hipotez yalnız EOS ile, log olasılık 0 ekleyerek uzar; başlangıçta tek hipotez; sonuç en yüksek toplam log olasılık.

- `beam_loop(step, h0, sos, eos, k, T)` — her batch elemanı ve her hipotez için ayrı `step` çağrısı.
- `beam_batched(step, h0, sos, eos, k, T)` — `(B·k)` hipotezi tek `step` çağrısıyla; aday skorları `(B, k·V)` → `topk`; geri izleme indeksleri `// V`, `% V`. Başlangıçta k − 1 hipotezin skoru −∞.
- İkisi de `(B, T)` token ve `(B,)` skor döndürür.

In [ ]:
%%writefile stage_e.py
import time
import torch


def beam_loop(step, h0, sos, eos, k, T):
    ...


def beam_batched(step, h0, sos, eos, k, T):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · BLEU: değiştirilmiş n-gram kesinliği ve kısalık cezası

Saf Python.

- `ngrams(tokens, n)` → sayım sözlüğü.
- `modified_precision(cand, refs, n)` → `(kırpılmış eşleşme, toplam)`: her n-gramın sayısı, herhangi bir referanstaki en büyük sayısıyla kırpılır.
- `brevity_penalty(c, r)` — $1$ ($c>r$) ya da $e^{1-r/c}$; r: c'ye en yakın referans uzunluğu (eşitlikte kısa olan).
- `bleu(cand, refs, N=4)` — $BP\cdot\exp(\frac1N\sum\log p_n)$; herhangi bir $p_n=0$ ise 0.

In [ ]:
%%writefile extra_x1.py
import math


def ngrams(tokens, n):
    ...


def modified_precision(cand, refs, n):
    ...


def brevity_penalty(c, r):
    ...


def bleu(cand, refs, N=4):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Kovalama: dolgu israfını ölçmek

NumPy.

- `padding_waste(lengths, batches)` — `batches`: indeks dizileri listesi; dolgu token'larının toplam token'lara (dolgulu) oranı.
- `random_batches(n, bs, rng)` ve `bucket_batches(lengths, bs, rng, chunk=50)` — veriyi rastgele karıştır, `chunk · bs`'lik parçalara böl, her parçayı uzunluğa göre sırala, batch'lere ayır, batch sırasını karıştır.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def padding_waste(lengths, batches):
    ...


def random_batches(n, bs, rng):
    ...


def bucket_batches(lengths, bs, rng, chunk=50):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Örneklenmiş softmax ve log Q düzeltmesi

NumPy. Çıkış katmanı `W (V, d)`, gizli durum `h (d,)`, hedef `t`.

- `full_grad(h, W, t)` — $\partial/\partial h$ of $-\log\mathrm{softmax}(Wh)_t$.
- `sampled_grad(h, W, t, S, logq, correct=True)` — örnek kümesi `S` (gürültü dağılımı q'dan, hedef hariç): logit'ler $\{w_t^\top h\}\cup\{w_s^\top h-\log q_s\cdot[\text{correct}]\}$ üzerinde softmax CE'nin h'ye göre gradyanı.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def full_grad(h, W, t):
    ...


def sampled_grad(h, W, t, S, logq, correct=True):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Bahdanau, Cho, Bengio (2015): hizalamayı birlikte öğrenmek

[Bahdanau, Cho, Bengio — Neural Machine Translation by Jointly Learning to Align and Translate (ICLR 2015)](https://arxiv.org/abs/1409.0473)

d aşamasındaki modeli gerçek bir paralel derleme (ör. Tatoeba İngilizce–Türkçe, ~100 bin cümle, BPE ile — M4.5) taşı: çift yönlü GRU kodlayıcı, toplamsal attention, maxout'lu çıktı (Bölüm A.2.2). RNNencdec ve RNNsearch'ü cümle uzunluğuna göre BLEU ile karşılaştır (makale Şekil 2: attention'sız modelin uzun cümlelerde çöküşü) ve birkaç cümle için hizalama haritası çiz (Şekil 3).


**Adımlar**

1. BLEU için x1'deki uygulamanı kullan (korpus düzeyi).
2. Uzunluk kovaları: 0–10, 10–20, 20–30, 30+ kelime.

**Kabul ölçütleri**

- Uzunluk arttıkça attention'sız modelin BLEU'su attention'lıdan hızla düşer.
- Hizalama haritaları kelime sırası farklarını (Türkçe SOV) yansıtır.

## Makale ödevi: Bengio ve ark. (2015): zamanlanmış örnekleme

[Bengio, Vinyals, Jaitly, Shazeer — Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks (NeurIPS 2015)](https://arxiv.org/abs/1506.03099)

Öğretmen zorlamasının "maruz kalma yanlılığını" (exposure bias) ölç: tarih çevirisinde kod çözücüye eğitimde gerçek önceki token yerine ε_i olasılıkla kendi tahminini ver; makaledeki doğrusal, üstel ve ters sigmoid azalma takvimlerini karşılaştır. Girdiye gürültü eklenmiş (yazım hatalı ay adları) bir test setinde tam eşleşme doğruluğunu ve hatanın ilk görüldüğü adımdan sonraki hata yayılımını raporla.


**Adımlar**

1. Ters sigmoid: $\epsilon_i=k/(k+\exp(i/k))$.

**Kabul ölçütleri**

- Zamanlanmış örnekleme gürültülü testte hata yayılımını azaltır.
- Takvimlerin karşılaştırma tablosu ve eğitim eğrileri.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 16](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Sutskever, Vinyals, Le — Sequence to Sequence Learning with Neural Networks (NeurIPS 2014)](https://arxiv.org/abs/1409.3215) — arXiv
- [Bahdanau, Cho, Bengio — Neural Machine Translation by Jointly Learning to Align and Translate (ICLR 2015)](https://arxiv.org/abs/1409.0473) — arXiv
- [Luong, Pham, Manning — Effective Approaches to Attention-based NMT (EMNLP 2015)](https://arxiv.org/abs/1508.04025) — arXiv
- [Bengio ve ark. — Scheduled Sampling for Sequence Prediction with RNNs (NeurIPS 2015)](https://arxiv.org/abs/1506.03099) — arXiv
- [Jean ve ark. — On Using Very Large Target Vocabulary for NMT (ACL 2015)](https://arxiv.org/abs/1412.2007) — arXiv
- [Papineni ve ark. — BLEU (ACL 2002)](https://aclanthology.org/P02-1040/) — ©
- [Stanford CS224n — Lecture 7: Machine Translation, Seq2Seq, Attention](https://web.stanford.edu/class/cs224n/) — ©
- [d2l.ai — 10.7 Seq2Seq, 10.8 Beam Search, 11.4 Bahdanau Attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/bahdanau-attention.html) — CC-BY-SA 4.0